# 02. QAT FP32 ONNX → Static INT8 ONNX v1

이 노트북은 15번 실험의 첫 번째 실제 판정 단계다.

00에서 만든 QAT 후보 `.pth` 4개는 **양자화 오차에 적응하도록 추가 학습된 FP32 모델**이다. 01에서는 그 `.pth`들을 일반 CLRKDNet 구조로 로드해서 **FP32 ONNX**로 내보냈고, PyTorch와 ONNX 출력이 같은지 확인했다.

02에서는 그 QAT FP32 ONNX 4개를 실제 **static INT8 ONNX**로 바꾼다. 그리고 세 가지 비교를 동시에 한다.

1. `QAT FP32 → QAT INT8`: 양자화가 QAT 모델을 얼마나 깨뜨렸는가.
2. `Original FP32 → QAT FP32`: QAT 학습 자체가 원래 모델에서 얼마나 멀어졌는가.
3. `Original FP32 → QAT INT8`: 최종 배포 후보가 12번에서 검증한 기준 모델과 얼마나 비슷한가.

즉, 이 노트북의 질문은 하나다.

> QAT를 거친 뒤 static INT8로 바꾸면, 07 decoder와 08 steering까지 보존하면서 Pi에 올릴 후보가 생기는가?


## QDQ static INT8를 한 번 더 직관적으로 정리

ONNX Runtime의 static quantization은 크게 두 표현이 있다.

- `QOperator`: `Conv` 같은 연산 자체를 `QLinearConv` 같은 quantized op로 바꾼다.
- `QDQ`: 그래프 안에 `QuantizeLinear`와 `DequantizeLinear` 노드를 넣어서, 어떤 텐서가 INT8 스케일로 표현되어야 하는지 표시한다.

QDQ는 겉으로 보면 `DequantizeLinear`가 있으니 “다시 FP32로 돌아가는 것 아닌가?”처럼 보일 수 있다. 하지만 ONNX Runtime은 QDQ 패턴을 보고 내부 최적화에서 INT8 kernel로 실행할 수 있다. 즉 QDQ는 **런타임에게 양자화 경계를 알려주는 그래프 표현**에 가깝다.

이번 02에서는 후보를 많이 늘리지 않는다. 11/11b에서 이미 여러 조합을 탐색했고 모두 의미 보존이 실패했기 때문이다. 여기서는 QAT 효과만 보기 위해 한 가지 레시피만 쓴다.

- format: `QDQ`
- activation: `QUInt8`
- weight: `QInt8`
- calibration: `MinMax`
- quantized op types: `Conv`, `MatMul`, `Gemm`

여기서 “전체 구조 static INT8”라는 말은 ONNX Runtime이 지원하는 주요 무거운 연산을 가능한 한 전부 static quantization한다는 뜻이다. ONNX graph 안의 모든 작은 보조 연산까지 억지로 INT8이 되는 것은 아니다. 그건 정상이다.


In [1]:
from pathlib import Path
import json
import time
import platform
import traceback

import numpy as np
import pandas as pd
import cv2
import onnx
import onnxruntime as ort

from tqdm.auto import tqdm
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)

print("python:", platform.python_version())
print("platform:", platform.platform())
print("onnx:", onnx.__version__)
print("onnxruntime:", ort.__version__)


python: 3.10.18
platform: Windows-10-10.0.26200-SP0
onnx: 1.21.0
onnxruntime: 1.23.2


In [2]:
# ----- Path configuration -----
EXP15 = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery")
EXP12 = EXP15.parent / "12_clrkdnet_supervised_rebuild"

ORIGINAL_FP32_ONNX = EXP12 / "review_outputs" / "09_onnx_export_parity_v1" / "models" / "MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx"
ORIGINAL_FP32_DATA = ORIGINAL_FP32_ONNX.with_name(ORIGINAL_FP32_ONNX.name + ".data")

QAT_FP32_DIR = EXP15 / "models" / "fp32_onnx"
INT8_OUT_DIR = EXP15 / "models" / "int8_onnx"

PKG10 = EXP12 / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
NOTEBOOK10 = EXP12 / "notebooks" / "10_pi_runtime_latency_sequence_validation_v1.ipynb"

REVIEW_OUT = EXP15 / "review_outputs" / "02_qat_static_int8_quantization_v1"
TABLES_DIR = REVIEW_OUT / "tables"
REPORTS_DIR = REVIEW_OUT / "reports"
VIS_DIR = REVIEW_OUT / "visuals"

for p in [INT8_OUT_DIR, REVIEW_OUT, TABLES_DIR, REPORTS_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    "qat_layer4_only",
    "qat_backbone_only",
    "qat_backbone_neck_only",
    "qat_full_model",
]

required = {
    "original_fp32_onnx": ORIGINAL_FP32_ONNX,
    "original_fp32_data": ORIGINAL_FP32_DATA,
    "qat_fp32_dir": QAT_FP32_DIR,
    "pkg10": PKG10,
    "notebook10": NOTEBOOK10,
}
for name, path in required.items():
    print(f"{name}: {path} exists={path.exists()}")
    assert path.exists(), f"Missing {name}: {path}"

for name in CANDIDATES:
    onnx_path = QAT_FP32_DIR / f"{name}_fp32.onnx"
    data_path = onnx_path.with_name(onnx_path.name + ".data")
    print(name, onnx_path.exists(), data_path.exists())
    assert onnx_path.exists(), onnx_path
    assert data_path.exists(), data_path


original_fp32_onnx: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx exists=True
original_fp32_data: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx.data exists=True
qat_fp32_dir: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\models\fp32_onnx exists=True
pkg10: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg exists=True
notebook10: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\notebooks\10_pi_runtime

## 실험 knob

처음 Run All이 너무 오래 걸리면 아래 limit을 줄이면 된다. 기본값은 11/11b와 비슷하게 잡았다.

- `CALIB_RECORD_LIMIT=None`: 10 package 안의 중복 제거 이미지 전체를 calibration에 사용한다.
- `PARITY_RECORD_LIMIT=None`: val/field3/holdout parity sample 전체를 raw/decode 비교에 사용한다.
- `SEQUENCE_RECORD_LIMIT=160`: field3 sequence 중 앞 160장을 steering 비교에 사용한다.
- `LATENCY_RECORD_LIMIT=120`: latency 측정은 120장으로 한다.

중요한 점: calibration에는 label이 필요 없다. 이미지를 넣어 activation range만 측정한다.


In [3]:
# ----- Experiment knobs -----
CALIB_RECORD_LIMIT = None
PARITY_RECORD_LIMIT = None
SEQUENCE_RECORD_LIMIT = 160
LATENCY_RECORD_LIMIT = 120

ORT_THREADS_LOCAL = None
ORT_WARMUP_RUNS = 5

QUANT_FORMAT = QuantFormat.QDQ
ACTIVATION_TYPE = QuantType.QUInt8
WEIGHT_TYPE = QuantType.QInt8
CALIB_METHOD = CalibrationMethod.MinMax
PER_CHANNEL = False
OP_TYPES_TO_QUANTIZE = ["Conv", "MatMul", "Gemm"]

QUANT_DECODE_COUNT_MISMATCH_LIMIT = 0
QUANT_STEER_MODE_MISMATCH_LIMIT = 0
QUANT_MEAN_LANE_DIST_LIMIT_PX = 2.0
QUANT_MAX_STEER_DIFF_LIMIT = 0.05

FINAL_DECODE_COUNT_MISMATCH_LIMIT = 0
FINAL_STEER_MODE_MISMATCH_LIMIT = 0
FINAL_MEAN_LANE_DIST_LIMIT_PX = 2.0
FINAL_MAX_STEER_DIFF_LIMIT = 0.05

print("CALIB_RECORD_LIMIT:", CALIB_RECORD_LIMIT)
print("PARITY_RECORD_LIMIT:", PARITY_RECORD_LIMIT)
print("SEQUENCE_RECORD_LIMIT:", SEQUENCE_RECORD_LIMIT)
print("LATENCY_RECORD_LIMIT:", LATENCY_RECORD_LIMIT)
print("OP_TYPES_TO_QUANTIZE:", OP_TYPES_TO_QUANTIZE)


CALIB_RECORD_LIMIT: None
PARITY_RECORD_LIMIT: None
SEQUENCE_RECORD_LIMIT: 160
LATENCY_RECORD_LIMIT: 120
OP_TYPES_TO_QUANTIZE: ['Conv', 'MatMul', 'Gemm']


## 10번 runtime core 재사용

02에서도 decoder와 steering을 새로 invent하지 않는다.

10번 노트북에서 이미 Pi용으로 정리했던 standalone runtime core를 그대로 읽어와서 사용한다. 여기에는 다음이 들어 있다.

- 12번 학습과 동일한 preprocess
- 07 official-overlap decoder의 numpy port
- 08 steering postprocess의 numpy port

이렇게 해야 02의 INT8 평가가 “새 평가 코드”가 아니라, 실제 Pi 배포 코드와 같은 인터페이스를 타게 된다.


In [4]:
# ----- Minimal globals required by the 10 notebook runtime core -----
IS_PI = False
RAW_W, RAW_H = 1296, 972
CUT_HEIGHT = 445
IMG_W, IMG_H = 800, 320
NUM_PRIORS = 192
OUTPUT_DIM = 78
N_OFFSETS = 72
N_STRIPS = N_OFFSETS - 1
SAMPLE_Y = list(range(971, 444, -20))
IMAGE_CENTER_X = RAW_W / 2.0
DEFAULT_PAIR_BONUS_PX = 60.0
DEFAULT_PI_ORT_THREADS = 4
REQUIRE_SCIPY_FOR_DECODER = True
ORT_WARMUP_RUNS = int(ORT_WARMUP_RUNS)

def fs_path(path):
    path = Path(path)
    s = str(path)
    if platform.system().lower().startswith("win"):
        try:
            s = str(path.resolve())
        except Exception:
            s = str(path)
        if not s.startswith("\\\\?\\"):
            s = "\\\\?\\" + s
    return s

def exists_fs(path):
    try:
        return Path(path).exists()
    except OSError:
        return Path(fs_path(path)).exists()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def imread_bgr(path):
    data = np.fromfile(fs_path(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def imwrite_bgr(path, img):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(Path(path).suffix or ".jpg", img)
    if not ok:
        raise RuntimeError(f"cv2.imencode failed: {path}")
    buf.tofile(fs_path(path))

try:
    from scipy.interpolate import InterpolatedUnivariateSpline
    HAS_SCIPY = True
except Exception as exc:
    HAS_SCIPY = False
    if REQUIRE_SCIPY_FOR_DECODER:
        raise RuntimeError("SciPy is required for official-compatible Lane.to_array spline resampling.") from exc

nb10 = json.loads(NOTEBOOK10.read_text(encoding="utf-8"))
runtime_core = None
for cell in nb10["cells"]:
    if cell.get("cell_type") == "code":
        src = "".join(cell.get("source", []))
        if src.lstrip().startswith("# ----- Contracts and ONNX Runtime -----"):
            runtime_core = src
            break
assert runtime_core is not None, "10 notebook runtime core cell was not found."
exec(compile(runtime_core, "10_runtime_core", "exec"), globals())
print("Loaded runtime core from:", NOTEBOOK10)
print("HAS_SCIPY:", HAS_SCIPY)


Loaded runtime core from: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\notebooks\10_pi_runtime_latency_sequence_validation_v1.ipynb
HAS_SCIPY: True


In [5]:
# ----- Load records and contracts from the 10 package -----
decode_contract, driving_contract, _, _ = load_package_contracts(PKG10)
records = pd.read_csv(PKG10 / "t" / "records_manifest.csv")
records["image_path"] = records["image_rel"].apply(lambda rel: str(PKG10 / rel))

parity_records = records[records["role"] == "parity"].sort_values(["set", "order"]).copy()
sequence_records = records[records["role"] == "sequence"].sort_values(["order"]).copy()

if PARITY_RECORD_LIMIT is not None:
    parity_records = parity_records.head(int(PARITY_RECORD_LIMIT)).copy()
if SEQUENCE_RECORD_LIMIT is not None:
    sequence_records = sequence_records.head(int(SEQUENCE_RECORD_LIMIT)).copy()

calib_records = records.drop_duplicates("image_rel").sort_values(["set", "role", "order"]).copy()
if CALIB_RECORD_LIMIT is not None:
    calib_records = calib_records.head(int(CALIB_RECORD_LIMIT)).copy()

latency_records = sequence_records.copy()
if LATENCY_RECORD_LIMIT is not None:
    latency_records = latency_records.head(int(LATENCY_RECORD_LIMIT)).copy()

print("records total:", len(records))
print("calibration records:", len(calib_records))
print("parity records:", len(parity_records))
print("sequence records:", len(sequence_records))
print("latency records:", len(latency_records))
print(records.groupby(["set", "role"]).size())


records total: 356
calibration records: 356
parity records: 116
sequence records: 160
latency records: 120
set      role    
field3   parity       80
         sequence    240
holdout  parity       24
val      parity       12
dtype: int64


## Calibration reader

ONNX Runtime static quantization API는 `CalibrationDataReader`라는 객체를 요구한다.

이 객체는 학습 데이터로 역전파를 하는 것이 아니다. 그저 이미지를 하나씩 읽어서 preprocess한 뒤, ONNX 모델에 넣어볼 수 있는 입력 tensor를 반환한다. Runtime은 그 입력들을 통과시키면서 각 activation tensor의 min/max 범위를 기록하고, 그 범위를 기준으로 `scale`과 `zero_point`를 정한다.


In [6]:
class ImageCalibrationDataReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name):
        self.image_paths = list(image_paths)
        self.input_name = input_name
        self._iter = None

    def get_next(self):
        if self._iter is None:
            self._iter = iter(self.image_paths)
        try:
            p = next(self._iter)
        except StopIteration:
            return None
        bgr = imread_bgr(p)
        return {self.input_name: preprocess_bgr_for_model(bgr)}

    def rewind(self):
        self._iter = None

def make_session_from_model(model_path, intra_op_num_threads=ORT_THREADS_LOCAL):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def warmup_session(session, input_name, image_paths, runs=ORT_WARMUP_RUNS):
    if len(image_paths) == 0:
        return
    bgr = imread_bgr(image_paths[0])
    inp = preprocess_bgr_for_model(bgr)
    for _ in range(int(runs)):
        session.run(None, {input_name: inp})

def run_raw_from_session(session, input_name, output_name, bgr):
    inp = preprocess_bgr_for_model(bgr)
    out = session.run([output_name], {input_name: inp})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

def file_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)

def remove_existing_onnx(path):
    path = Path(path)
    for p in [path, path.with_name(path.name + ".data")]:
        if p.exists():
            p.unlink()


## QAT 후보 4개를 static INT8 ONNX로 변환

여기서 생성되는 파일은 실제 배포 후보다.

`qat_layer4_only_fp32.onnx` → `qat_layer4_only_static_qdq_u8s8.onnx` 같은 식으로 하나씩 만들어진다.

주의할 점은, 이 단계가 정확도 판정이 아니라 **파일 생성 단계**라는 것이다. 성공적으로 `.onnx`가 만들어졌다고 해서 바로 쓸 수 있는 모델이라는 뜻은 아니다. 그 다음 셀들에서 decoder와 steering까지 비교해야 한다.


In [7]:
def quantize_one_qat_candidate(candidate_name):
    source_model = QAT_FP32_DIR / f"{candidate_name}_fp32.onnx"
    output_model = INT8_OUT_DIR / f"{candidate_name}_static_qdq_u8s8.onnx"
    assert source_model.exists(), source_model
    assert source_model.with_name(source_model.name + ".data").exists(), source_model.with_name(source_model.name + ".data")

    probe_session, input_name, _ = make_session_from_model(source_model)
    image_paths = calib_records["image_path"].tolist()
    reader = ImageCalibrationDataReader(image_paths, input_name)

    remove_existing_onnx(output_model)
    t0 = time.perf_counter()
    status = "ok"
    error = ""
    try:
        quantize_static(
            model_input=fs_path(source_model),
            model_output=fs_path(output_model),
            calibration_data_reader=reader,
            quant_format=QUANT_FORMAT,
            op_types_to_quantize=OP_TYPES_TO_QUANTIZE,
            per_channel=PER_CHANNEL,
            activation_type=ACTIVATION_TYPE,
            weight_type=WEIGHT_TYPE,
            calibrate_method=CALIB_METHOD,
            use_external_data_format=False,
        )
        assert output_model.exists(), output_model
        q_model = onnx.load(fs_path(output_model), load_external_data=True)
        onnx.checker.check_model(q_model)
    except Exception as exc:
        status = "error"
        error = "".join(traceback.format_exception_only(type(exc), exc)).strip()
    sec = time.perf_counter() - t0

    return {
        "candidate": candidate_name,
        "status": status,
        "error": error,
        "source_fp32_onnx": str(source_model),
        "int8_onnx": str(output_model),
        "quant_sec": float(sec),
        "fp32_size_mb": file_mb(source_model) + file_mb(source_model.with_name(source_model.name + ".data")),
        "int8_size_mb": file_mb(output_model) if output_model.exists() else np.nan,
        "format": "QDQ",
        "activation_type": "QUInt8",
        "weight_type": "QInt8",
        "calibrate_method": "MinMax",
        "per_channel": bool(PER_CHANNEL),
        "op_types_to_quantize": ",".join(OP_TYPES_TO_QUANTIZE),
    }

quant_rows = []
for candidate in CANDIDATES:
    print("\n=== quantize", candidate, "===")
    row = quantize_one_qat_candidate(candidate)
    print(row)
    quant_rows.append(row)

quant_df = pd.DataFrame(quant_rows)
quant_df.to_csv(TABLES_DIR / "quantization_generation.csv", index=False, encoding="utf-8-sig")
display(quant_df)



=== quantize qat_layer4_only ===


{'candidate': 'qat_layer4_only', 'status': 'ok', 'error': '', 'source_fp32_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\fp32_onnx\\qat_layer4_only_fp32.onnx', 'int8_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\int8_onnx\\qat_layer4_only_static_qdq_u8s8.onnx', 'quant_sec': 65.42466260000037, 'fp32_size_mb': 44.29033088684082, 'int8_size_mb': 11.524415016174316, 'format': 'QDQ', 'activation_type': 'QUInt8', 'weight_type': 'QInt8', 'calibrate_method': 'MinMax', 'per_channel': False, 'op_types_to_quantize': 'Conv,MatMul,Gemm'}

=== quantize qat_backbone_only ===


{'candidate': 'qat_backbone_only', 'status': 'ok', 'error': '', 'source_fp32_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\fp32_onnx\\qat_backbone_only_fp32.onnx', 'int8_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\int8_onnx\\qat_backbone_only_static_qdq_u8s8.onnx', 'quant_sec': 50.88796250000087, 'fp32_size_mb': 44.29044246673584, 'int8_size_mb': 11.52441692352295, 'format': 'QDQ', 'activation_type': 'QUInt8', 'weight_type': 'QInt8', 'calibrate_method': 'MinMax', 'per_channel': False, 'op_types_to_quantize': 'Conv,MatMul,Gemm'}

=== quantize qat_backbone_neck_only ===


{'candidate': 'qat_backbone_neck_only', 'status': 'ok', 'error': '', 'source_fp32_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\fp32_onnx\\qat_backbone_neck_only_fp32.onnx', 'int8_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\int8_onnx\\qat_backbone_neck_only_static_qdq_u8s8.onnx', 'quant_sec': 51.23359899999923, 'fp32_size_mb': 44.29072570800781, 'int8_size_mb': 11.524415016174316, 'format': 'QDQ', 'activation_type': 'QUInt8', 'weight_type': 'QInt8', 'calibrate_method': 'MinMax', 'per_channel': False, 'op_types_to_quantize': 'Conv,MatMul,Gemm'}

=== quantize qat_full_model ===


{'candidate': 'qat_full_model', 'status': 'ok', 'error': '', 'source_fp32_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\fp32_onnx\\qat_full_model_fp32.onnx', 'int8_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\15_clrkdnet_qat_int8_recovery\\models\\int8_onnx\\qat_full_model_static_qdq_u8s8.onnx', 'quant_sec': 53.74678209999911, 'fp32_size_mb': 44.29027557373047, 'int8_size_mb': 11.52441692352295, 'format': 'QDQ', 'activation_type': 'QUInt8', 'weight_type': 'QInt8', 'calibrate_method': 'MinMax', 'per_channel': False, 'op_types_to_quantize': 'Conv,MatMul,Gemm'}


,candidate,status,error,source_fp32_onnx,int8_onnx,quant_sec,fp32_size_mb,int8_size_mb,format,activation_type,weight_type,calibrate_method,per_channel,op_types_to_quantize
0,qat_layer4_only,ok,,~\02_Projects\University\26-1_Em...,~\02_Projects\University\26-1_Em...,65.424663,44.290331,11.524415,QDQ,QUInt8,QInt8,MinMax,False,"Conv,MatMul,Gemm"
1,qat_backbone_only,ok,,~\02_Projects\University\26-1_Em...,~\02_Projects\University\26-1_Em...,50.887963,44.290442,11.524417,QDQ,QUInt8,QInt8,MinMax,False,"Conv,MatMul,Gemm"
2,qat_backbone_neck_only,ok,,~\02_Projects\University\26-1_Em...,~\02_Projects\University\26-1_Em...,51.233599,44.290726,11.524415,QDQ,QUInt8,QInt8,MinMax,False,"Conv,MatMul,Gemm"
3,qat_full_model,ok,,~\02_Projects\University\26-1_Em...,~\02_Projects\University\26-1_Em...,53.746782,44.290276,11.524417,QDQ,QUInt8,QInt8,MinMax,False,"Conv,MatMul,Gemm"


## 비교 함수

Raw output 숫자만 보는 것으로는 부족하다. lane detector는 raw tensor에서 끝나는 모델이 아니라, 그 뒤에 07 decoder와 08 steering이 붙어야 실제 주행 의미가 생긴다.

그래서 비교를 3층으로 나눈다.

1. Raw tensor: ONNX 출력 자체가 얼마나 다른가.
2. Decoded lane: `get_lanes`에 해당하는 decoder를 통과한 레인 개수와 좌표가 같은가.
3. Steering sequence: field3 sequence에서 memory를 누적했을 때 steering mode와 steer 값이 같은가.


In [8]:
def lane_pair_distance(lane_a, lane_b):
    pa = np.asarray(lane_a["points"], dtype=np.float32)
    pb = np.asarray(lane_b["points"], dtype=np.float32)
    if len(pa) == 0 or len(pb) == 0:
        return np.nan
    dists = []
    for x, y in pa:
        j = int(np.argmin(np.abs(pb[:, 1] - y)))
        if abs(float(pb[j, 1] - y)) <= 1.0:
            dists.append(abs(float(pb[j, 0] - x)))
    return float(np.mean(dists)) if dists else np.nan

def compare_lane_sets(base_lanes, test_lanes):
    count_mismatch = int(len(base_lanes) != len(test_lanes))
    pair_dists = []
    for a, b in zip(base_lanes, test_lanes):
        d = lane_pair_distance(a, b)
        if np.isfinite(d):
            pair_dists.append(d)
    return {
        "base_count": len(base_lanes),
        "test_count": len(test_lanes),
        "count_mismatch": count_mismatch,
        "mean_pair_dist_px": float(np.mean(pair_dists)) if pair_dists else np.nan,
        "max_pair_dist_px": float(np.max(pair_dists)) if pair_dists else np.nan,
    }

def model_registry_from_quant_df(quant_df):
    registry = {"original_fp32": ORIGINAL_FP32_ONNX}
    for candidate in CANDIDATES:
        registry[f"{candidate}__qat_fp32"] = QAT_FP32_DIR / f"{candidate}_fp32.onnx"
        registry[f"{candidate}__qat_int8"] = INT8_OUT_DIR / f"{candidate}_static_qdq_u8s8.onnx"
    return registry

def comparison_specs_from_quant_df(quant_df):
    specs = []
    for candidate in CANDIDATES:
        specs.append({
            "candidate": candidate,
            "axis": "qat_fp32_vs_qat_int8",
            "base_model": f"{candidate}__qat_fp32",
            "test_model": f"{candidate}__qat_int8",
            "meaning": "양자화 자체가 QAT 모델을 얼마나 깨뜨렸는지",
        })
        specs.append({
            "candidate": candidate,
            "axis": "original_fp32_vs_qat_fp32",
            "base_model": "original_fp32",
            "test_model": f"{candidate}__qat_fp32",
            "meaning": "QAT 학습 자체가 원본 모델에서 얼마나 멀어졌는지",
        })
        specs.append({
            "candidate": candidate,
            "axis": "original_fp32_vs_qat_int8",
            "base_model": "original_fp32",
            "test_model": f"{candidate}__qat_int8",
            "meaning": "최종 배포 후보가 기존 검증 기준 모델과 얼마나 비슷한지",
        })
    return specs


## 모델 세션 생성과 warmup

ONNX Runtime latency는 첫 실행이 유독 느릴 수 있다. 그래프 최적화와 메모리 할당이 첫 실행에 섞이기 때문이다. 그래서 모든 session은 평가 전에 몇 번 warmup한다.


In [9]:
model_registry = model_registry_from_quant_df(quant_df)
model_sessions = {}
latency_image_paths = latency_records["image_path"].tolist()

for model_name, model_path in model_registry.items():
    if not Path(model_path).exists():
        print("skip missing model:", model_name, model_path)
        continue
    session, input_name, output_name = make_session_from_model(model_path)
    warmup_session(session, input_name, latency_image_paths)
    model_sessions[model_name] = {
        "path": Path(model_path),
        "session": session,
        "input_name": input_name,
        "output_name": output_name,
    }
    print("session ready:", model_name, model_path)

assert "original_fp32" in model_sessions
for candidate in CANDIDATES:
    assert f"{candidate}__qat_fp32" in model_sessions, candidate
    if (INT8_OUT_DIR / f"{candidate}_static_qdq_u8s8.onnx").exists():
        assert f"{candidate}__qat_int8" in model_sessions, candidate


session ready: original_fp32 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx
session ready: qat_layer4_only__qat_fp32 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\models\fp32_onnx\qat_layer4_only_fp32.onnx
session ready: qat_layer4_only__qat_int8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\models\int8_onnx\qat_layer4_only_static_qdq_u8s8.onnx
session ready: qat_backbone_only__qat_fp32 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\models\fp32_onnx\qat_backbone_only_fp32.onnx
session ready: qat_backbone_only__qat_int8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recove

In [10]:
def run_model_raw(model_name, bgr):
    m = model_sessions[model_name]
    return run_raw_from_session(m["session"], m["input_name"], m["output_name"], bgr)

def evaluate_comparison(spec):
    base_model = spec["base_model"]
    test_model = spec["test_model"]
    axis = spec["axis"]
    candidate = spec["candidate"]
    raw_rows = []
    decode_rows = []
    steer_rows = []

    if base_model not in model_sessions or test_model not in model_sessions:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for _, rec in tqdm(parity_records.iterrows(), total=len(parity_records), desc=f"{candidate} {axis} raw/decode"):
        bgr = imread_bgr(rec["image_path"])
        base_raw = run_model_raw(base_model, bgr)
        test_raw = run_model_raw(test_model, bgr)
        diff = np.abs(base_raw - test_raw)

        base_lanes = decode_raw_to_lanes(base_raw, decode_contract)
        test_lanes = decode_raw_to_lanes(test_raw, decode_contract)
        lane_cmp = compare_lane_sets(base_lanes, test_lanes)

        raw_rows.append({
            "candidate": candidate,
            "axis": axis,
            "base_model": base_model,
            "test_model": test_model,
            "key": rec["key"],
            "set": rec["set"],
            "role": rec["role"],
            "order": int(rec["order"]),
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
            "p99_abs_diff": float(np.quantile(diff, 0.99)),
        })
        decode_rows.append({
            "candidate": candidate,
            "axis": axis,
            "base_model": base_model,
            "test_model": test_model,
            "key": rec["key"],
            "set": rec["set"],
            "role": rec["role"],
            "order": int(rec["order"]),
            **lane_cmp,
        })

    base_mem = init_drive_memory()
    test_mem = init_drive_memory()
    for _, rec in tqdm(sequence_records.iterrows(), total=len(sequence_records), desc=f"{candidate} {axis} steering"):
        bgr = imread_bgr(rec["image_path"])
        base_raw = run_model_raw(base_model, bgr)
        test_raw = run_model_raw(test_model, bgr)
        base_lanes = decode_raw_to_lanes(base_raw, decode_contract)
        test_lanes = decode_raw_to_lanes(test_raw, decode_contract)
        base_row = update_drive(base_lanes, base_mem, driving_contract)
        test_row = update_drive(test_lanes, test_mem, driving_contract)
        steer_rows.append({
            "candidate": candidate,
            "axis": axis,
            "base_model": base_model,
            "test_model": test_model,
            "key": rec["key"],
            "order": int(rec["order"]),
            "base_mode": base_row["effective_mode"],
            "test_mode": test_row["effective_mode"],
            "mode_mismatch": int(base_row["effective_mode"] != test_row["effective_mode"]),
            "base_steer": float(base_row["steer_norm"]),
            "test_steer": float(test_row["steer_norm"]),
            "steer_abs_diff": abs(float(base_row["steer_norm"]) - float(test_row["steer_norm"])),
            "center_abs_diff": abs(float(base_row["smoothed_center_x"]) - float(test_row["smoothed_center_x"])),
            "heading_abs_diff": abs(float(base_row["smoothed_heading"]) - float(test_row["smoothed_heading"])),
        })
    return pd.DataFrame(raw_rows), pd.DataFrame(decode_rows), pd.DataFrame(steer_rows)

comparison_specs = comparison_specs_from_quant_df(quant_df)
raw_parts, decode_parts, steer_parts = [], [], []

for spec in comparison_specs:
    if spec["test_model"] not in model_sessions:
        print("skip comparison because test model is missing:", spec)
        continue
    raw_part, decode_part, steer_part = evaluate_comparison(spec)
    if not raw_part.empty:
        raw_parts.append(raw_part)
    if not decode_part.empty:
        decode_parts.append(decode_part)
    if not steer_part.empty:
        steer_parts.append(steer_part)

raw_df = pd.concat(raw_parts, ignore_index=True) if raw_parts else pd.DataFrame()
decode_df = pd.concat(decode_parts, ignore_index=True) if decode_parts else pd.DataFrame()
steer_df = pd.concat(steer_parts, ignore_index=True) if steer_parts else pd.DataFrame()

raw_df.to_csv(TABLES_DIR / "comparison_raw_parity.csv", index=False, encoding="utf-8-sig")
decode_df.to_csv(TABLES_DIR / "comparison_decode_parity.csv", index=False, encoding="utf-8-sig")
steer_df.to_csv(TABLES_DIR / "comparison_steering_parity.csv", index=False, encoding="utf-8-sig")

print("raw rows:", len(raw_df))
print("decode rows:", len(decode_df))
print("steer rows:", len(steer_df))


qat_layer4_only qat_fp32_vs_qat_int8 raw/decode:   0%|          | 0/116 [00:00<?, ?it/s]

qat_layer4_only qat_fp32_vs_qat_int8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

qat_layer4_only original_fp32_vs_qat_fp32 raw/decode:   0%|          | 0/116 [00:00<?, ?it/s]

qat_layer4_only original_fp32_vs_qat_fp32 steering:   0%|          | 0/160 [00:00<?, ?it/s]

qat_layer4_only original_fp32_vs_qat_int8 raw/decode:   0%|          | 0/116 [00:00<?, ?it/s]

...

qat_full_model original_fp32_vs_qat_fp32 raw/decode:   0%|          | 0/116 [00:00<?, ?it/s]

qat_full_model original_fp32_vs_qat_fp32 steering:   0%|          | 0/160 [00:00<?, ?it/s]

qat_full_model original_fp32_vs_qat_int8 raw/decode:   0%|          | 0/116 [00:00<?, ?it/s]

qat_full_model original_fp32_vs_qat_int8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

raw rows: 1392
decode rows: 1392
steer rows: 1920


## Latency 측정

여기서는 로컬 CPU에서 상대 비교를 본다. Pi latency는 이후 Pi 검증 노트북에서 다시 봐야 한다.

그래도 로컬에서 미리 보는 이유는 단순하다.

- INT8가 실제로 FP32보다 빨라지는지
- QAT 후보별 INT8 속도 차이가 있는지
- decode/steering overhead가 모델 추론보다 큰지 작은지

이 셀의 latency는 Pi의 최종 성능을 보장하지 않는다. 하지만 후보를 거르는 1차 정보로는 충분하다.


In [11]:
def measure_model_latency(model_name):
    if model_name not in model_sessions:
        return pd.DataFrame()
    m = model_sessions[model_name]
    rows = []
    mem = init_drive_memory()
    for _, rec in tqdm(latency_records.iterrows(), total=len(latency_records), desc=f"latency {model_name}"):
        bgr = imread_bgr(rec["image_path"])
        t0 = time.perf_counter()
        inp = preprocess_bgr_for_model(bgr)
        t1 = time.perf_counter()
        raw = m["session"].run([m["output_name"]], {m["input_name"]: inp})[0][0].astype(np.float32)
        t2 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw, decode_contract)
        t3 = time.perf_counter()
        drive_row = update_drive(lanes, mem, driving_contract)
        t4 = time.perf_counter()
        rows.append({
            "model": model_name,
            "key": rec["key"],
            "order": int(rec["order"]),
            "preprocess_ms": (t1 - t0) * 1000,
            "inference_ms": (t2 - t1) * 1000,
            "decode_ms": (t3 - t2) * 1000,
            "steering_ms": (t4 - t3) * 1000,
            "pipeline_ms": (t4 - t0) * 1000,
            "lanes": len(lanes),
            "mode": drive_row["effective_mode"],
            "steer_norm": float(drive_row["steer_norm"]),
        })
    return pd.DataFrame(rows)

latency_parts = []
for model_name in ["original_fp32"] + [f"{c}__qat_fp32" for c in CANDIDATES] + [f"{c}__qat_int8" for c in CANDIDATES]:
    if model_name in model_sessions:
        latency_parts.append(measure_model_latency(model_name))

latency_df = pd.concat(latency_parts, ignore_index=True) if latency_parts else pd.DataFrame()
latency_df.to_csv(TABLES_DIR / "model_latency.csv", index=False, encoding="utf-8-sig")
print("latency rows:", len(latency_df))


latency original_fp32:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_layer4_only__qat_fp32:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_backbone_only__qat_fp32:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_backbone_neck_only__qat_fp32:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_full_model__qat_fp32:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_layer4_only__qat_int8:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_backbone_only__qat_int8:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_backbone_neck_only__qat_int8:   0%|          | 0/120 [00:00<?, ?it/s]

latency qat_full_model__qat_int8:   0%|          | 0/120 [00:00<?, ?it/s]

latency rows: 1080


## 결과 요약과 후보 선택

요약표는 세 비교축을 모두 포함한다.

특히 봐야 할 것은 두 줄이다.

- `qat_fp32_vs_qat_int8`: 이게 좋으면 QAT가 INT8 변환을 견디는 것이다.
- `original_fp32_vs_qat_int8`: 이게 좋으면 실제로 12번 FP32 기준 모델을 대체할 후보가 되는 것이다.

둘 다 좋아야 Pi로 가져갈 명분이 생긴다.


In [12]:
def q95(series):
    series = series.dropna()
    return float(series.quantile(0.95)) if len(series) else np.nan

summary_rows = []
for spec in comparison_specs:
    candidate = spec["candidate"]
    axis = spec["axis"]
    raw_sub = raw_df[(raw_df["candidate"] == candidate) & (raw_df["axis"] == axis)] if not raw_df.empty else pd.DataFrame()
    dec_sub = decode_df[(decode_df["candidate"] == candidate) & (decode_df["axis"] == axis)] if not decode_df.empty else pd.DataFrame()
    ste_sub = steer_df[(steer_df["candidate"] == candidate) & (steer_df["axis"] == axis)] if not steer_df.empty else pd.DataFrame()
    if raw_sub.empty and dec_sub.empty and ste_sub.empty:
        continue

    decode_count_mismatch = int(dec_sub["count_mismatch"].sum()) if not dec_sub.empty else np.nan
    steer_mode_mismatch = int(ste_sub["mode_mismatch"].sum()) if not ste_sub.empty else np.nan
    decode_mean_pair_dist = float(dec_sub["mean_pair_dist_px"].dropna().mean()) if not dec_sub.empty else np.nan
    decode_max_pair_dist = float(dec_sub["max_pair_dist_px"].dropna().max()) if not dec_sub.empty else np.nan
    steer_max_abs_diff = float(ste_sub["steer_abs_diff"].max()) if not ste_sub.empty else np.nan
    steer_mean_abs_diff = float(ste_sub["steer_abs_diff"].mean()) if not ste_sub.empty else np.nan

    if axis == "qat_fp32_vs_qat_int8":
        pass_gate = (
            decode_count_mismatch <= QUANT_DECODE_COUNT_MISMATCH_LIMIT
            and steer_mode_mismatch <= QUANT_STEER_MODE_MISMATCH_LIMIT
            and (not np.isfinite(decode_mean_pair_dist) or decode_mean_pair_dist <= QUANT_MEAN_LANE_DIST_LIMIT_PX)
            and (not np.isfinite(steer_max_abs_diff) or steer_max_abs_diff <= QUANT_MAX_STEER_DIFF_LIMIT)
        )
    elif axis == "original_fp32_vs_qat_int8":
        pass_gate = (
            decode_count_mismatch <= FINAL_DECODE_COUNT_MISMATCH_LIMIT
            and steer_mode_mismatch <= FINAL_STEER_MODE_MISMATCH_LIMIT
            and (not np.isfinite(decode_mean_pair_dist) or decode_mean_pair_dist <= FINAL_MEAN_LANE_DIST_LIMIT_PX)
            and (not np.isfinite(steer_max_abs_diff) or steer_max_abs_diff <= FINAL_MAX_STEER_DIFF_LIMIT)
        )
    else:
        pass_gate = np.nan

    summary_rows.append({
        "candidate": candidate,
        "axis": axis,
        "base_model": spec["base_model"],
        "test_model": spec["test_model"],
        "pass_gate": pass_gate,
        "raw_max_abs_diff": float(raw_sub["max_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "raw_mean_abs_diff_max": float(raw_sub["mean_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "raw_p99_abs_diff_max": float(raw_sub["p99_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "decode_count_mismatch": decode_count_mismatch,
        "decode_mean_pair_dist_px": decode_mean_pair_dist,
        "decode_max_pair_dist_px": decode_max_pair_dist,
        "steering_mode_mismatch": steer_mode_mismatch,
        "steering_max_abs_diff": steer_max_abs_diff,
        "steering_mean_abs_diff": steer_mean_abs_diff,
        "meaning": spec["meaning"],
    })

comparison_summary = pd.DataFrame(summary_rows)
comparison_summary.to_csv(TABLES_DIR / "comparison_summary.csv", index=False, encoding="utf-8-sig")

lat_summary_rows = []
if not latency_df.empty:
    for model_name, sub in latency_df.groupby("model"):
        mean_pipeline = float(sub["pipeline_ms"].mean())
        lat_summary_rows.append({
            "model": model_name,
            "pipeline_mean_ms": mean_pipeline,
            "pipeline_p50_ms": float(sub["pipeline_ms"].quantile(0.50)),
            "pipeline_p95_ms": q95(sub["pipeline_ms"]),
            "inference_mean_ms": float(sub["inference_ms"].mean()),
            "decode_mean_ms": float(sub["decode_ms"].mean()),
            "steering_mean_ms": float(sub["steering_ms"].mean()),
            "fps_from_pipeline_mean": float(1000.0 / mean_pipeline) if mean_pipeline > 0 else np.nan,
        })
latency_summary = pd.DataFrame(lat_summary_rows)
latency_summary.to_csv(TABLES_DIR / "latency_summary.csv", index=False, encoding="utf-8-sig")

display(comparison_summary.sort_values(["candidate", "axis"]))
display(latency_summary.sort_values("pipeline_mean_ms") if not latency_summary.empty else latency_summary)


,candidate,axis,base_model,test_model,pass_gate,raw_max_abs_diff,raw_mean_abs_diff_max,raw_p99_abs_diff_max,decode_count_mismatch,decode_mean_pair_dist_px,decode_max_pair_dist_px,steering_mode_mismatch,steering_max_abs_diff,steering_mean_abs_diff,meaning
7,qat_backbone_neck_only,original_fp32_vs_qat_fp32,original_fp32,qat_backbone_neck_only__qat_fp32,NaN,1040.425171,2.130439,8.076944,7,11.229476,822.892237,5,0.171354,0.010654,QAT 학습 자체가 원본 모델에서 얼마나 멀어졌는지
8,qat_backbone_neck_only,original_fp32_vs_qat_int8,original_fp32,qat_backbone_neck_only__qat_int8,False,1390.170532,2.848204,28.866238,7,58.671681,1246.068481,8,0.124987,0.011441,최종 배포 후보가 기존 검증 기준 모델과 얼마나 비슷한지
6,qat_backbone_neck_only,qat_fp32_vs_qat_int8,qat_backbone_neck_only__qat_fp32,qat_backbone_neck_only__qat_int8,False,539.183350,1.358958,19.450500,2,51.171809,1227.366211,9,0.167017,0.019741,양자화 자체가 QAT 모델을 얼마나 깨뜨렸는지
4,qat_backbone_only,original_fp32_vs_qat_fp32,original_fp32,qat_backbone_only__qat_fp32,NaN,1466.812500,2.993988,34.943871,6,24.321090,822.866412,3,0.079138,0.005741,QAT 학습 자체가 원본 모델에서 얼마나 멀어졌는지
5,qat_backbone_only,original_fp32_vs_qat_int8,original_fp32,qat_backbone_only__qat_int8,False,1536.336304,4.027980,22.939255,10,46.912790,860.599476,11,0.163907,0.014458,최종 배포 후보가 기존 검증 기준 모델과 얼마나 비슷한지
3,qat_backbone_only,qat_fp32_vs_qat_int8,qat_backbone_only__qat_fp32,qat_backbone_only__qat_int8,False,1529.601562,3.796880,14.222828,7,35.910096,860.028304,10,0.167889,0.011586,양자화 자체가 QAT 모델을 얼마나 깨뜨렸는지
10,qat_full_model,original_fp32_vs_qat_fp32,original_fp32,qat_full_model__qat_fp32,NaN,1453.580566,2.963670,15.609613,6,19.031489,574.677781,1,0.029603,0.004201,QAT 학습 자체가 원본 모델에서 얼마나 멀어졌는지
11,qat_full_model,original_fp32_vs_qat_int8,original_fp32,qat_full_model__qat_int8,False,1389.770996,2.841992,46.940453,13,65.230187,859.703681,7,0.163478,0.013327,최종 배포 후보가 기존 검증 기준 모델과 얼마나 비슷한지
9,qat_full_model,qat_fp32_vs_qat_int8,qat_full_model__qat_fp32,qat_full_model__qat_int8,False,1156.309326,2.369381,36.979847,9,63.011694,863.748325,6,0.145692,0.011875,양자화 자체가 QAT 모델을 얼마나 깨뜨렸는지
1,qat_layer4_only,original_fp32_vs_qat_fp32,original_fp32,qat_layer4_only__qat_fp32,NaN,1494.328979,3.042380,42.988132,7,19.939577,824.051216,7,0.138633,0.007520,QAT 학습 자체가 원본 모델에서 얼마나 멀어졌는지


,model,pipeline_mean_ms,pipeline_p50_ms,pipeline_p95_ms,inference_mean_ms,decode_mean_ms,steering_mean_ms,fps_from_pipeline_mean
8,qat_layer4_only__qat_int8,28.107129,23.28245,61.443180,24.523577,0.523052,0.144083,35.578162
6,qat_full_model__qat_int8,29.380234,24.25140,64.945370,25.601626,0.539203,0.141390,34.036488
3,qat_backbone_only__qat_fp32,42.917906,37.18920,41.731795,39.611342,0.538893,0.152176,23.300298
7,qat_layer4_only__qat_fp32,43.327548,37.39155,50.219775,40.016750,0.564253,0.146575,23.080004
0,original_fp32,48.454625,38.76225,92.680085,44.818257,0.724828,0.141565,20.637865
4,qat_backbone_only__qat_int8,50.518771,46.28500,97.491770,45.244227,0.867660,0.357846,19.794623
1,qat_backbone_neck_only__qat_fp32,51.791507,37.76830,116.565425,47.949292,0.774780,0.140515,19.308185
2,qat_backbone_neck_only__qat_int8,58.002328,51.17600,90.258925,52.658154,1.110482,0.138217,17.240687
5,qat_full_model__qat_fp32,62.586873,40.00425,143.199590,58.775119,0.543727,0.146057,15.977792


In [13]:
selected_rows = []
for candidate in CANDIDATES:
    qrow = comparison_summary[(comparison_summary["candidate"] == candidate) & (comparison_summary["axis"] == "qat_fp32_vs_qat_int8")]
    frow = comparison_summary[(comparison_summary["candidate"] == candidate) & (comparison_summary["axis"] == "original_fp32_vs_qat_int8")]
    if qrow.empty or frow.empty:
        continue
    quant_pass = bool(qrow.iloc[0]["pass_gate"])
    final_pass = bool(frow.iloc[0]["pass_gate"])
    model_name = f"{candidate}__qat_int8"
    lat_row = latency_summary[latency_summary["model"] == model_name] if not latency_summary.empty else pd.DataFrame()
    selected_rows.append({
        "candidate": candidate,
        "quantization_pass": quant_pass,
        "final_similarity_pass": final_pass,
        "selected_for_pi": bool(quant_pass and final_pass),
        "int8_model": str(INT8_OUT_DIR / f"{candidate}_static_qdq_u8s8.onnx"),
        "pipeline_mean_ms": float(lat_row.iloc[0]["pipeline_mean_ms"]) if not lat_row.empty else np.nan,
        "fps_from_pipeline_mean": float(lat_row.iloc[0]["fps_from_pipeline_mean"]) if not lat_row.empty else np.nan,
    })

selection_df = pd.DataFrame(selected_rows)
selection_df.to_csv(TABLES_DIR / "candidate_selection.csv", index=False, encoding="utf-8-sig")
display(selection_df.sort_values(["selected_for_pi", "pipeline_mean_ms"], ascending=[False, True]))

selected_for_pi_df = selection_df[selection_df["selected_for_pi"] == True].copy()
if len(selected_for_pi_df):
    selected_for_pi = selected_for_pi_df.sort_values("pipeline_mean_ms").iloc[0].to_dict()
    print("Selected for Pi validation:")
    print(json.dumps(selected_for_pi, indent=2, ensure_ascii=False))
else:
    selected_for_pi = None
    print("No candidate passed both gates. Inspect comparison_summary.csv for near-misses.")


,candidate,quantization_pass,final_similarity_pass,selected_for_pi,int8_model,pipeline_mean_ms,fps_from_pipeline_mean
0,qat_layer4_only,False,False,False,~\02_Projects\University\26-1_Em...,28.107129,35.578162
3,qat_full_model,False,False,False,~\02_Projects\University\26-1_Em...,29.380234,34.036488
1,qat_backbone_only,False,False,False,~\02_Projects\University\26-1_Em...,50.518771,19.794623
2,qat_backbone_neck_only,False,False,False,~\02_Projects\University\26-1_Em...,58.002328,17.240687


No candidate passed both gates. Inspect comparison_summary.csv for near-misses.


## 시각 검수 이미지

정량 지표가 통과해도, 레인이 눈으로 봤을 때 이상하게 뒤틀리는 경우가 있을 수 있다. 그래서 후보 하나를 골라 original FP32와 QAT INT8 overlay를 나란히 그린다.

- 초록: original FP32 decoded lane
- 분홍: QAT INT8 decoded lane

통과 후보가 있으면 그 후보를 그린다. 통과 후보가 없으면 최종 비교축에서 가장 가까워 보이는 후보를 하나 골라 near-miss 검수용으로 그린다.


In [14]:
def choose_visual_candidate():
    if selected_for_pi is not None:
        return selected_for_pi["candidate"]
    sub = comparison_summary[comparison_summary["axis"] == "original_fp32_vs_qat_int8"].copy()
    if sub.empty:
        return CANDIDATES[0]
    sub["score"] = (
        sub["decode_count_mismatch"].fillna(9999) * 1000
        + sub["steering_mode_mismatch"].fillna(9999) * 100
        + sub["decode_mean_pair_dist_px"].fillna(9999)
        + sub["steering_max_abs_diff"].fillna(9999) * 10
    )
    return str(sub.sort_values("score").iloc[0]["candidate"])

def draw_lanes(img, lanes, color, thickness=2):
    out = img.copy()
    for lane in lanes:
        pts = np.asarray(lane["points"], dtype=np.float32)
        if len(pts) < 2:
            continue
        pts_i = pts.astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(out, [pts_i], isClosed=False, color=color, thickness=thickness, lineType=cv2.LINE_AA)
    return out

def make_overlay_sheet(candidate, max_frames=12):
    test_model = f"{candidate}__qat_int8"
    if test_model not in model_sessions:
        print("missing test model for visual:", test_model)
        return None
    rows = []
    sample = parity_records.head(max_frames).copy()
    for _, rec in sample.iterrows():
        bgr = imread_bgr(rec["image_path"])
        base_raw = run_model_raw("original_fp32", bgr)
        test_raw = run_model_raw(test_model, bgr)
        base_lanes = decode_raw_to_lanes(base_raw, decode_contract)
        test_lanes = decode_raw_to_lanes(test_raw, decode_contract)
        canvas = bgr.copy()
        canvas = draw_lanes(canvas, base_lanes, (0, 220, 0), thickness=2)
        canvas = draw_lanes(canvas, test_lanes, (255, 0, 255), thickness=2)
        title = f"{rec['set']} #{int(rec['order'])} | green=original fp32, magenta={candidate} int8"
        cv2.putText(canvas, title, (20, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(canvas, title, (20, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
        rows.append(canvas)
    if not rows:
        return None
    sheet = np.vstack(rows)
    out_path = VIS_DIR / f"{candidate}_original_vs_int8_overlay.jpg"
    imwrite_bgr(out_path, sheet)
    return out_path

visual_candidate = choose_visual_candidate()
visual_path = make_overlay_sheet(visual_candidate)
print("visual_candidate:", visual_candidate)
print("visual_path:", visual_path)


visual_candidate: qat_layer4_only
visual_path: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\review_outputs\02_qat_static_int8_quantization_v1\visuals\qat_layer4_only_original_vs_int8_overlay.jpg


## Final report

이 report가 02의 결론 파일이다.

가능한 결론은 셋 중 하나다.

1. `selected_for_pi_validation`이 있다: 03에서 Pi INT8 검증으로 넘어간다.
2. 통과 후보는 없지만 near-miss가 있다: 기준을 완화할지, QAT scope/epoch를 조정할지 판단한다.
3. 모두 크게 깨진다: QAT-lite로는 부족하고 다른 양자화 전략 또는 모델 경량화가 필요하다.


In [15]:
calibration_sets = {
    f"{set_name}/{role_name}": int(count)
    for (set_name, role_name), count in calib_records.groupby(["set", "role"]).size().items()
}

report = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "02_static_int8_quantize_qat_onnx_v1.ipynb",
    "original_fp32_onnx": str(ORIGINAL_FP32_ONNX),
    "qat_fp32_dir": str(QAT_FP32_DIR),
    "int8_out_dir": str(INT8_OUT_DIR),
    "source_package": str(PKG10),
    "quantization_recipe": {
        "format": "QDQ",
        "activation_type": "QUInt8",
        "weight_type": "QInt8",
        "calibrate_method": "MinMax",
        "per_channel": bool(PER_CHANNEL),
        "op_types_to_quantize": OP_TYPES_TO_QUANTIZE,
        "note": "Static quantization uses calibration images only; labels/GT are not used.",
    },
    "calibration": {
        "records": int(len(calib_records)),
        "sets": calibration_sets,
        "requires_gt": False,
    },
    "thresholds": {
        "quant_decode_count_mismatch_limit": QUANT_DECODE_COUNT_MISMATCH_LIMIT,
        "quant_steer_mode_mismatch_limit": QUANT_STEER_MODE_MISMATCH_LIMIT,
        "quant_mean_lane_dist_limit_px": QUANT_MEAN_LANE_DIST_LIMIT_PX,
        "quant_max_steer_diff_limit": QUANT_MAX_STEER_DIFF_LIMIT,
        "final_decode_count_mismatch_limit": FINAL_DECODE_COUNT_MISMATCH_LIMIT,
        "final_steer_mode_mismatch_limit": FINAL_STEER_MODE_MISMATCH_LIMIT,
        "final_mean_lane_dist_limit_px": FINAL_MEAN_LANE_DIST_LIMIT_PX,
        "final_max_steer_diff_limit": FINAL_MAX_STEER_DIFF_LIMIT,
    },
    "quantization_generation": quant_df.to_dict(orient="records"),
    "comparison_summary": comparison_summary.to_dict(orient="records"),
    "latency_summary": latency_summary.to_dict(orient="records") if not latency_summary.empty else [],
    "selection": selection_df.to_dict(orient="records") if not selection_df.empty else [],
    "selected_for_pi_validation": selected_for_pi,
    "visual_overlay": str(visual_path) if visual_path is not None else None,
    "next": [
        "If selected_for_pi_validation is not null, create/run 03 Pi INT8 runtime validation.",
        "Pi validation must check local INT8 vs Pi INT8 parity and Pi latency/fps.",
        "If no candidate passes, inspect near-miss rows before changing QAT scope/epochs or quantization recipe.",
    ],
}
write_json(REPORTS_DIR / "qat_static_int8_quantization_report.json", report)

if selected_for_pi is not None:
    write_json(REPORTS_DIR / "selected_int8_candidate_for_pi.json", selected_for_pi)
    print("selected candidate written:", REPORTS_DIR / "selected_int8_candidate_for_pi.json")
else:
    print("No selected candidate file written because no candidate passed both gates.")

print("report:", REPORTS_DIR / "qat_static_int8_quantization_report.json")


No selected candidate file written because no candidate passed both gates.
report: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery\review_outputs\02_qat_static_int8_quantization_v1\reports\qat_static_int8_quantization_report.json


## 02 결과 읽는 법

Run All 후에는 먼저 아래 파일을 보면 된다.

- `review_outputs/02_qat_static_int8_quantization_v1/tables/comparison_summary.csv`
- `review_outputs/02_qat_static_int8_quantization_v1/tables/candidate_selection.csv`
- `review_outputs/02_qat_static_int8_quantization_v1/tables/latency_summary.csv`
- `review_outputs/02_qat_static_int8_quantization_v1/reports/qat_static_int8_quantization_report.json`

판단 순서는 다음과 같다.

1. `qat_fp32_vs_qat_int8`가 통과하는지 본다. 이게 실패하면 QAT 모델도 INT8 변환을 못 견딘 것이다.
2. `original_fp32_vs_qat_int8`가 통과하는지 본다. 이게 통과해야 12번 FP32 기준 모델을 대체할 수 있다.
3. 둘 다 통과한 후보 중 latency가 가장 낮은 것을 Pi 검증 후보로 삼는다.

이 단계에서 통과 후보가 나오면, 다음은 Pi에서 같은 INT8 ONNX를 돌려 `local INT8 ≈ Pi INT8`와 실제 latency를 확인하는 단계다.
